# MLflow — DAIS 2026 Runbook

Story: Three agents in production: **Refund**, **Complaint**, **Operational Supervisor** (fans out to 6 KAs + 3 Genie spaces).

**Message**: One system, governed by UC. No extra services.

> Docs: [MLflow GenAI](https://mlflow.org/docs/latest/llms/genai/index.html) · [Tracing](https://mlflow.org/docs/latest/llms/tracing/index.html) · [Evaluation](https://mlflow.org/docs/latest/llms/genai-evaluation/index.html) · [Prompt Registry](https://mlflow.org/docs/latest/llms/prompt-engineering/index.html)

### Pre-flight

- Send one message through the Operational Dashboard (so the **Sessions** tab has at least one entry to point at)
- Run the next cell for fresh URLs.

In [ ]:
# Pre-flight: print fresh URLs for the demo.
# Re-run any time IDs go stale (e.g. after redeploy).

def _autodetect_caspers_catalog():
    """Find the most recently-deployed Casper's catalog via its uc_state table.

    Every Casper's deployment creates `<catalog>._internal_state.resources`,
    so the catalog with the most recently altered such table is the freshest
    deployment in this workspace.  Falls back to `caspersdev` if the system
    tables aren't queryable or no Casper's deployment is found.
    """
    try:
        rows = spark.sql("""
            SELECT table_catalog FROM system.information_schema.tables
            WHERE table_schema = '_internal_state' AND table_name = 'resources'
            ORDER BY last_altered DESC LIMIT 1
        """).collect()
        return rows[0].table_catalog if rows else "caspersdev"
    except Exception:
        return "caspersdev"

try:
    _detected = _autodetect_caspers_catalog()
    # Recreate the widget so the displayed default always reflects the freshest
    # deployment — `dbutils.widgets.text` does not reliably update the displayed
    # value when the widget already exists from a previous run.
    try:
        dbutils.widgets.remove("CATALOG")
    except Exception:
        pass
    dbutils.widgets.text("CATALOG", _detected, "UC Catalog")
    CATALOG = dbutils.widgets.get("CATALOG") or _detected
except Exception:
    CATALOG = "caspersdev"

import json
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
host = w.config.host.rstrip("/")
me = spark.sql("SELECT current_user()").collect()[0][0]

print(f"Catalog: {CATALOG}")
print(f"Host:    {host}")
print(f"User:    {me}\n")


def _exp_url(name: str) -> str:
    """Resolve an experiment by name and return a clickable URL, or '(not created yet)'.

    Uses the Databricks SDK's experiments API rather than mlflow, so this
    works on serverless runtimes that don't ship mlflow by default.  The
    only mlflow-only section in this runbook is the prompt registry listing
    below, which imports mlflow lazily and degrades gracefully if missing.
    """
    try:
        resp = w.experiments.get_by_name(name)
        exp = getattr(resp, "experiment", None) or resp
        eid = getattr(exp, "experiment_id", None)
        if eid:
            return f"{host}/ml/experiments/{eid}"
    except Exception:
        pass
    return "(not created yet — fire one request to the endpoint or run the eval task)"


def _endpoint_url(name: str) -> str:
    try:
        ep = w.serving_endpoints.get(name)
        state = getattr(getattr(ep, "state", None), "ready", "?")
        return f"{host}/ml/endpoints/{name}  [{state}]"
    except Exception:
        return f"{name}  (not deployed in this target)"


# ── Agent serving endpoints ──────────────────────────────────────────────────
print("Agent endpoints")
print("=" * 80)
print(f"  Refund Agent:        {_endpoint_url(f'{CATALOG}_refund_agent')}")
print(f"  Complaint Agent:     {_endpoint_url(f'{CATALOG}_complaint_agent')}")

# Operational Supervisor endpoint name is dynamic — read it from uc_state.
supervisor_endpoint = ""
try:
    df = spark.sql(f"""
        SELECT resource_data FROM {CATALOG}._internal_state.resources
        WHERE resource_type = 'multi_agent_supervisors'
        ORDER BY created_at DESC LIMIT 1
    """)
    rows = df.collect()
    if rows:
        supervisor_endpoint = json.loads(rows[0].resource_data).get("endpoint_name", "")
except Exception as e:
    print(f"  Could not read supervisor endpoint from uc_state: {e}")
print(f"  Operational Sup'r:   {_endpoint_url(supervisor_endpoint) if supervisor_endpoint else '(not deployed)'}")

# ── Per-agent dev/prod experiments ───────────────────────────────────────────
print("\nDevelopment experiments (where evaluation runs land)")
print("=" * 80)
print(f"  Refund dev:    {_exp_url(f'/Shared/{CATALOG}_refund_agent_dev')}")
print(f"  Refund prod:   {_exp_url(f'/Shared/{CATALOG}_refund_agent_prod')}")
print(f"  Complaint dev: {_exp_url(f'/Shared/{CATALOG}_complaint_agent_dev')}")
print(f"  Complaint prod:{_exp_url(f'/Shared/{CATALOG}_complaint_agent_prod')}")
if supervisor_endpoint:
    mas_id = supervisor_endpoint.replace("-endpoint", "")
    print(f"  Supervisor:    {_exp_url(f'/Users/{me}/{mas_id}-dev-experiment')}")

# ── Production / serving experiments (where live traces land) ────────────────
print("\nProduction experiments (auto-created by Model Serving — every live request lands here)")
print("=" * 80)
for label, ep in [
    ("Refund",     f"{CATALOG}_refund_agent"),
    ("Complaint",  f"{CATALOG}_complaint_agent"),
    ("Supervisor", supervisor_endpoint),
]:
    if ep:
        print(f"  {label:<11} {_exp_url(f'/Serving/{ep}')}")

# ── UC-managed evaluation datasets ───────────────────────────────────────────
print("\nUC-managed evaluation datasets (one per agent)")
print("=" * 80)
_uc_datasets = [
    ("Supervisor", "operational_supervisor_eval_dataset", "Evaluation task"),
    ("Refund",     "refund_agent_eval_dataset",          "Refund_Setup task"),
    ("Complaint",  "complaint_agent_eval_dataset",       "Complaint_Setup task"),
]
for _label, _table, _origin in _uc_datasets:
    _full = f"{CATALOG}.evaluations.{_table}"
    try:
        _n = spark.sql(f"SELECT COUNT(*) AS n FROM {_full}").collect()[0].n
        print(f"  {_label:<11} {_full}  ({_n} rows)")
        print(f"              {host}/explore/data/{CATALOG}/evaluations/{_table}")
    except Exception:
        print(f"  {_label:<11} {_full}  (not created — run {_origin})")

# ── Prompt registry ──────────────────────────────────────────────────────────
# Prompts are stored in UC under <catalog>.prompts. They are NOT browsable
# as tables in Catalog Explorer (they're a different UC entity type — `SHOW
# TABLES IN prompts` returns zero).  The way to inspect prompt versions in
# the workspace UI is via the **Prompts tab on each Experiment that loaded
# the prompt** — those URLs are already printed above:
#   Refund dev/prod        → loads `refund_system`         (Prompts tab)
#   Complaint dev/prod     → loads `complaint_system`      (Prompts tab)
#   Supervisor experiment  → loads `supervisor_instructions` (Prompts tab)
print("\nPrompt registry")
print("=" * 80)
try:
    import mlflow
    # Without databricks-uc registry, search_prompts hits workspace MLflow
    # and the catalog/schema filter returns nothing even if prompts exist in UC.
    mlflow.set_registry_uri("databricks-uc")
    prompts = mlflow.genai.search_prompts(
        filter_string=f"catalog = '{CATALOG}' AND schema = 'prompts'"
    )
    if not prompts:
        print(f"  No prompts under {CATALOG}.prompts — has the `all` target been deployed?")
    else:
        for p in sorted(prompts, key=lambda x: x.name):
            print(f"  {p.name}")
        print(f"\n  To see versions + tags + diffs:")
        print(f"    Open any of the experiment URLs printed above → click the 'Prompts' tab.")
        print(f"    Each experiment's Prompts tab lists the prompts loaded by that endpoint")
        print(f"    (via mlflow.genai.load_prompt) and their version history.")
except ImportError:
    print(f"  mlflow not installed in this runtime — install it to list prompts:")
    print(f"    %pip install -qq 'mlflow-skinny[databricks]'   # then restart kernel and re-run")
    print(f"  (You can still browse them via the Prompts tab on any experiment URL above.)")
except Exception as e:
    print(f"  Could not list prompts under {CATALOG}.prompts: {type(e).__name__}: {e}")
    print(f"  (You can still browse them via the Prompts tab on any experiment URL above.)")

![MLflow Architecture](assets/mlflow_gen_arch.png)

### Tracing

**Show:**
- `Experiments`→ `mas_xxx experiment` → recent trace → waterfall (Genie SQL 10–30 s · KA 2–8 s · supervisor synth 3–6 s — bottleneck visible without instrumentation)
- `Experiments`→ `{catalog}_refund_agent_prod` → recent trace → LangGraph nodes + every tool call + every LLM call as separate spans

**Message:** every agent traced by default — MAS by Agent Bricks, refund + complaint via `mlflow.langchain.autolog()`



### Prompts

**Show:**
- MLflow experiment -> prompts

**Message:** prompts are governed UC artifacts; updates flow without rebuilding models. Refund agent loads at request-time; supervisor at construction-time; complaint is baked into the DSPy Signature at model-log time (registry entry is audit-only — re-baking requires a redeploy). KAs intentionally don't register prompts — their `instructions` are short literals managed in the stage notebook directly.



### Evaluation

**Show:**
- `/Users/{me}/{mas_id}-dev-experiment` → latest `ceo-supervisor-full-eval` run
- 10 questions × 4 scorers: `ExpectationsGuidelines`, `Safety`, `routing_accuracy` (custom `@scorer`), `cites_specific_data` (custom `@scorer`)
- Same shape on `{cat}-refund-agent-eval` and `{cat}-complaint-agent-eval`

**Three UC-managed datasets, one per agent:**

| Agent | UC table | Built by |
|---|---|---|
| Supervisor | `{CATALOG}.evaluations.operational_supervisor_eval_dataset` | `Evaluation` |
| Refund | `{CATALOG}.evaluations.refund_agent_eval_dataset` | `Refund_Setup` |
| Complaint | `{CATALOG}.evaluations.complaint_agent_eval_dataset` | `Complaint_Setup` |


**Message:** behavioral guidelines (not factual)



### Production monitoring

**Show:** prod experiment for any agent → **Assessments** column → sort by `safety < 1` or `operational_quality = 0` to surface caught failures.

**Scorers (100% sampling, registered as last step of each stage):**
- All three: `safety`, `relevance_to_query`, `operational_quality`
- Refund: `refund_policy_compliance`
- Complaint: `decision_quality`, `refund_reason`
- Supervisor: `routing_accuracy`, `cites_specific_data`

**Message:** same judges that gate dev eval keep running on every live request.


### Unity AI Gateway